# 01 - Run the whole-tissue segmentation engine

Take one large 3D stack plus the two Cellpose models you trained in EpiCycle Steps 2-3, and produce a single labelled volume with a per-cell feature table: tile the image, segment each tile, reconstruct it in 3D, stitch the tiles, clean up, and measure.

The whole-tissue engine (`packages/epicycle_scaledseg`) is Step 5 of the workflow. It runs from this notebook or from the command line (`epicycle-scaledseg run --help`), in the same environment as the plugin.

**Before you start - hardware**

- **An NVIDIA GPU (CUDA) is required.** The segmentation stage refuses to run with zero GPUs.
- **RAM at least the size of your input image.** The input is read whole before it is tiled; one tile needs roughly 25 GB at 1024 px.
- **Disk about 3-4x the input size** (tiles, flows, probabilities and the stitched outputs).
- Tested on Linux GPU clusters (SLURM) and on one small run on Windows. Whole-tissue scale has only been tested on Linux.

Fill in the **EDIT** cells (marked below), then Run All.

In [ ]:
import sys
from pathlib import Path

# Make `packages.epicycle_scaledseg` importable when the notebook is opened from the
# repository root or from tutorials/segmentation/ (not needed if you ran `pip install -e .`).
for _p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_p / "packages" / "epicycle_scaledseg").is_dir():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

## 1. Paths and input

The paper's whole-tissue runs were on data resampled to isotropic voxels. Anisotropic data works, but the features come out in the units you state in `VOXEL_SIZE_UM` below.

In [ ]:
from pathlib import Path

# ===== EDIT: paths =====
INPUT_TIF = Path("EDIT_ME/your_stack.tif")   # a ZCYX, CZYX or ZYX TIFF; None if tiles already exist
WORK_DIR = Path("EDIT_ME/work")              # everything is written under WORK_DIR / OUTPUT_NAME
OUTPUT_NAME = "scaledseg_output"

# The axes of INPUT_TIF. "auto" reads the TIFF metadata; set it explicitly ("CZYX", "ZCYX", "ZYX")
# for thin stacks (Z <= number of channels) that carry no axes metadata.
INPUT_AXES = "auto"

# ===== EDIT: voxel size, REQUIRED =====
# Physical voxel size in micrometres as (z, y, x), e.g. (1.0, 0.3, 0.3).
# You can read it in EpiCycle Step 1 (the source's voxel size). Used for the cleanup report and the features.
VOXEL_SIZE_UM = None

In [ ]:
if VOXEL_SIZE_UM is None:
    print("WARNING: VOXEL_SIZE_UM is None.")
    print("  The cleanup report and every feature (areas, volumes, lengths, centroids) will be in")
    print("  VOXEL units, not micrometres. Voxels are rarely cubic, so shape features will be")
    print("  wrong for any analysis that assumes physical units. Set VOXEL_SIZE_UM above and re-run.")
else:
    print(f"Voxel size (z, y, x) in um: {VOXEL_SIZE_UM}")

## 2. Which model files

The engine needs the **two files** you trained in EpiCycle, one per view bucket: `lateral` (used for XY slices) and `axial` (used for XZ and YZ slices). They have **no file extension**, they are not `.pth` files, and a pretrained name such as `cyto3` cannot be used - both paths must exist.

Where EpiCycle writes them:

- **Step 3 (the usual source):** `artifacts/step3/refinement/models/<name>/lateral` and `.../axial` (or, for a multi-run, `artifacts/step3/runs/<run_id>/refinement/models/<name>/`).
- **Step 2:** `artifacts/step2/runs/<run>/<bucket>/round_NN/model/`.
- **Use the same pair Step 4 shows.** Step 4 records the active pair in `artifacts/step4/step4_state.json`; open it to see which two files it uses.

**Channels.** Set `CHANNELS` exactly as Step 4 does: the channel index **plus one** (`[2, 1]` means "channel 2 is the signal, channel 1 is the second channel" in Cellpose's 1-based convention; `[0, 0]` for single-channel grayscale). The cell below prints the channels the model was trained with, when its `manifest.json` records them.

**Feed the same preprocessed image.** Step 1 preprocessing is applied to your extracted patches, not to the raw TIFF. If your models were trained on preprocessed patches, run `INPUT_TIF` through the same recipe first.

In [ ]:
import json

# ===== EDIT: the two model files (no extension) =====
LATERAL_MODEL = Path("EDIT_ME/artifacts/step3/refinement/models/your_model/lateral")
AXIAL_MODEL = Path("EDIT_ME/artifacts/step3/refinement/models/your_model/axial")

# ===== EDIT: Cellpose channels, index + 1 as in Step 4 =====
# Cells: CHANNELS = [2, 1] (membrane as the segmentation channel, nuclei as the second); nuclei: CHANNELS = [1, 2].
CHANNELS = [2, 1]
FLOW_THRESHOLD = 0.2          # only filters Cellpose's 2D masks, which the 3D reconstruction does not use; the paper's runs used 0.2
CELLPROB_THRESHOLD = 0.0

for label, path in (("lateral", LATERAL_MODEL), ("axial", AXIAL_MODEL)):
    assert path.exists(), f"{label} model not found: {path}  (edit LATERAL_MODEL / AXIAL_MODEL above)"
    manifest = path.parent / "manifest.json"
    if manifest.exists():
        try:
            m = json.loads(manifest.read_text(encoding="utf-8"))
            used = (m.get("training") or {}).get("channels_used", m.get("channels_used"))
            print(f"{label}: {path.name}  trained on channels: {used}")
        except Exception as exc:
            print(f"{label}: {path.name}  (manifest.json unreadable: {exc})")
    else:
        print(f"{label}: {path.name}  (no manifest.json next to it)")
print(f"CHANNELS used here: {CHANNELS}")

## 3. Hardware and tiling

`PATCH_SIZE` is the XY tile size in pixels and `OVERLAP` the fraction shared between neighbouring tiles. The segmentation batch size was tuned on cluster GPUs (1024): on an 8 GB card use `BATCH_SIZE = 16`.

In [ ]:
# ===== EDIT: hardware =====
PATCH_SIZE = 1024      # XY tile size in pixels (smaller = less RAM per tile, more seams)
OVERLAP = 0.25         # fraction shared between neighbouring tiles
N_GPUS = None          # None = detect from SLURM / CUDA; an integer to fix it
BATCH_SIZE = 16        # 16 for 8 GB cards; the cluster-tuned value is 1024

## 4. Mode and stages

`MODE = "full"` is the recommended path: each tile is reconstructed in 3D and the tiles are joined by an instance-preserving, full-resolution seam stitch. It writes `stage4_full_stitch/global_labels.tif` and `.npy`, `result_manifest.json` and `label_provenance.csv` (a per-cell list of the cells most likely to be mis-stitched), and it is the only mode that supports cleanup and features.

The alternative, `"coarse"`, stitches the tiles and then re-reconstructs a factor-2 downsampled copy. It is lossy, writes a different output, and has no cleanup or features. It is not run from this notebook; use `epicycle-scaledseg run --help` for it.

In [ ]:
MODE = "full"          # "full" (recommended). "coarse" (lossy) is available from the command line only.

RUN_TILING = INPUT_TIF is not None
RUN_SEGMENTATION = True
RUN_RECONSTRUCTION = True
RUN_STITCH = True
RUN_POSTPROCESS = True   # drop tiny fragments and detached debris, smooth each cell
RUN_FEATURES = True      # per-cell feature table

## 5. Advanced parameters

Change these only if you know why. Tiling: the Z range is trimmed using an intensity threshold on one channel. Stitch: cells in neighbouring tiles are merged when their overlap (IoU) passes `IOU_TAU` and shares at least `IOU_VMIN` voxels.

The z-range is taken from the membrane channel (`CHANNEL_INDEX = 1`) for both the cell run and the nuclei run, so both tilings are identical and the two label volumes line up for the reconcile notebook.

In [ ]:
# Tiling
Z_THRESHOLD_METHOD = "hard"   # "hard", "percentile" or "default"
Z_THRESHOLD_VALUE = 0.07
CHANNEL_INDEX = 1             # channel used for the Z-range detection (0-based)
AUTO_RAM_CAP = True           # shrink the XY tile so one tile fits this machine's RAM

# Reconstruction
RECON_MODE = "sequential"     # "sequential" is the safe default; "parallel" on a large node
SAVE_PROB3D = True
SAVE_FLOW3D = False
SAVE_BINARY = True

# Smooths the 2D probability maps before 3D reconstruction; these are the values used for the paper's whole-tissue runs; set PRE_SMOOTH = False to skip
PRE_SMOOTH = True
PRE_SMOOTH_SIGMA_PERP = 2.5     # perpendicular to the plane
PRE_SMOOTH_SIGMA_INPLANE = 1.0  # in-plane

# IoU seam stitch
IOU_TAU = 0.25
IOU_VMIN = 10
IOU_Z_BLOCK = 64
IOU_ONE_TO_MANY = "max_iou"
EMIT_PROVENANCE = True

# Cleanup (PP_) and features (FEAT_)
PP_MIN_VOXELS = 200           # drop cells smaller than this many voxels
PP_REMOVE_ISLANDS = True
PP_SMOOTH = True
PP_SMOOTH_SIGMA = 1.5
PP_DRY_RUN = False            # True = report what would be dropped, write nothing

FEAT_FEATURE_SET = "epicycle"   # the EpiCycle feature catalogue (same columns as Steps 3-4)
FEAT_INCLUDE_SURFACE = True     # marching-cubes surface family; the slow part (False = fast pass)
FEAT_INVERT_Z = False           # True if high Z is the basal side
FEAT_DRY_RUN = False            # True = report the cell count and size, write nothing
FEAT_CHANNELS = [(1, "mem")]    # (channel index in the input stack, column prefix) -> adds the mem_* intensity columns; None = shape only

## 6. Set up

In [ ]:
import logging
from packages.epicycle_scaledseg.utils.resources import get_allocated_gpus

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

n_gpus = N_GPUS or get_allocated_gpus()
if n_gpus < 1:
    raise SystemExit("No NVIDIA GPU found. The segmentation stage needs at least one CUDA GPU.")

work_dir = Path(WORK_DIR)
output_dir = work_dir / OUTPUT_NAME
log_dir = output_dir / "logs"
log_dir.mkdir(parents=True, exist_ok=True)
tiles_dir = output_dir / "tiles"
stitch_dir = output_dir / "stage4_full_stitch"

if RUN_TILING:
    assert Path(INPUT_TIF).exists(), f"INPUT_TIF not found: {INPUT_TIF}"
    tile_dirs = [str(work_dir)]   # placeholder; tiling creates the real tile directories
else:
    tile_dirs = sorted(str(p) for p in tiles_dir.iterdir() if p.is_dir() and not p.name.startswith("."))
    print(f"Found {len(tile_dirs)} existing tile directories in {tiles_dir}")

print(f"GPUs: {n_gpus}")
print(f"Output: {output_dir}")

## 7. Build and run the pipeline

Segmentation of every tile is the long step. A run can be resumed: re-running this cell skips finished tiles.

In [ ]:
from packages.epicycle_scaledseg.core.pipeline import EpiCyclePipeline
from packages.epicycle_scaledseg.core.tiling import TilingStage
from packages.epicycle_scaledseg.core.segmentation import SegmentationStage
from packages.epicycle_scaledseg.core.reconstruction import ReconstructionStage
from packages.epicycle_scaledseg.core.iou_stitch_stage import IoUStitchStage
from packages.epicycle_scaledseg.config.defaults import get_default_segment_params

pipeline = EpiCyclePipeline(work_dir=str(work_dir), n_gpus=n_gpus, log_dir=str(log_dir), mode=MODE)

if RUN_TILING:
    pipeline.add_stage(TilingStage(
        input_tif=str(INPUT_TIF), output_dir=str(tiles_dir),
        patch_size=PATCH_SIZE, overlap=OVERLAP,
        z_threshold_method=Z_THRESHOLD_METHOD, threshold_value=Z_THRESHOLD_VALUE,
        channel_index=CHANNEL_INDEX, auto_correct=False,
        input_axes=INPUT_AXES, auto_ram_cap=AUTO_RAM_CAP,
    ))

if RUN_SEGMENTATION:
    segment_params = get_default_segment_params()
    segment_params.update({
        "flow_threshold": FLOW_THRESHOLD, "cellprob_threshold": CELLPROB_THRESHOLD,
        "channels": CHANNELS, "batch_size": BATCH_SIZE,
    })
    pipeline.add_stage(SegmentationStage(
        lateral_model=str(LATERAL_MODEL), axial_model=str(AXIAL_MODEL),
        segment_params=segment_params,
        out_root=str(output_dir / "stage1_segmentation"), gpu_strategy="auto",
    ))

if RUN_RECONSTRUCTION:
    pipeline.add_stage(ReconstructionStage(
        seg_root=str(output_dir / "stage1_segmentation"),
        out_root=str(output_dir / "stage2_reconstruction"),
        mode=RECON_MODE, save_prob3d=SAVE_PROB3D, save_flow3d=SAVE_FLOW3D,
        pre_smooth_probs=PRE_SMOOTH, pre_smooth_sigma_perp=PRE_SMOOTH_SIGMA_PERP,
        pre_smooth_sigma_inplane=PRE_SMOOTH_SIGMA_INPLANE,
        save_binary=SAVE_BINARY, binary_erode=True, binary_kernel_size=3,
        binary_border_margin=2, binary_fill_holes=True,
    ))

if RUN_STITCH:
    pipeline.add_stage(IoUStitchStage(
        recon_root=str(output_dir / "stage2_reconstruction"), output_dir=str(stitch_dir),
        tau=IOU_TAU, vmin=IOU_VMIN, one_to_many=IOU_ONE_TO_MANY,
        z_block=IOU_Z_BLOCK, emit_provenance=EMIT_PROVENANCE,
    ))

print(f"Pipeline ({MODE} mode): {[s.name for s in pipeline.stages]}")
result = pipeline.run(tile_dirs)
print(result.summary())

## 8. Cleanup and smoothing

Drops tiny fragments and detached debris and smooths each cell. Writes `global_labels_clean.tif` / `.npy` next to the stitch, without touching the raw stitch. Set `PP_DRY_RUN = True` first to see how many cells each cut would drop.

In [ ]:
if RUN_POSTPROCESS:
    from packages.epicycle_scaledseg.core.postprocess_stage import PostprocessStage
    pp = PostprocessStage(
        input_dir=str(stitch_dir), output_dir=str(stitch_dir),
        remove_small=True, min_voxels=PP_MIN_VOXELS,
        remove_islands=PP_REMOVE_ISLANDS,
        smooth=PP_SMOOTH, smooth_sigma=PP_SMOOTH_SIGMA, preserve_topology=True,
        voxel_size_um=VOXEL_SIZE_UM, dry_run=PP_DRY_RUN,
    )
    print(pp.run([stitch_dir], n_gpus=n_gpus).summary())
else:
    print("Cleanup skipped (RUN_POSTPROCESS = False).")

## 9. Per-cell features

Measures every cleaned cell with the EpiCycle feature set (`FEAT_FEATURE_SET = "epicycle"`), writing `cell_features.csv` / `.parquet` and `feature_manifest.json` next to the stitch. The `epicycle` set also turns on the contact graph (`edges.csv`) and the quality score. Size the job first with `FEAT_DRY_RUN = True` and `FEAT_INCLUDE_SURFACE = False`, then re-run with both back to normal. `FEAT_CHANNELS` names which image channel (0-based array index) is sampled for the per-cell intensity features and the prefix its columns get (`mem` for membrane, giving `mem_intensity_mean` and so on). It must match the channel your images actually carry: the default `[(1, "mem")]` assumes the membrane is channel 1, as in the model cell's `CHANNELS` setting, and a second channel can be added as `(0, "nuc")`.

In [ ]:
if RUN_FEATURES:
    from packages.epicycle_scaledseg.core.feature_stage import FeatureStage
    feat = FeatureStage(
        input_dir=str(stitch_dir), output_dir=str(stitch_dir),
        prefer_clean=True, voxel_size_um=VOXEL_SIZE_UM,
        include_surface=FEAT_INCLUDE_SURFACE, invert_z=FEAT_INVERT_Z,
        dry_run=FEAT_DRY_RUN, feature_set=FEAT_FEATURE_SET,
        channels=FEAT_CHANNELS, tiles_dir=str(tiles_dir) if FEAT_CHANNELS else None,
    )
    print(feat.run([stitch_dir], n_gpus=n_gpus).summary())
else:
    print("Features skipped (RUN_FEATURES = False).")

## 10. Check the result

The final labels are `stage4_full_stitch/global_labels.tif` (raw stitch) and `global_labels_clean.tif` (after cleanup). `label_provenance.csv` lists, per cell, whether it touches a tile seam or corner; the lowest-`min_match_iou` seam cells are the ones to review first.

In [ ]:
import numpy as np
import pandas as pd

for name in ("global_labels.tif", "global_labels_clean.tif", "result_manifest.json",
             "label_provenance.csv", "cell_features.csv", "edges.csv"):
    p = stitch_dir / name
    print(f"{name:26s} {'present' if p.exists() else 'MISSING'}")

clean_npy = stitch_dir / "global_labels_clean.npy"
npy = clean_npy if clean_npy.exists() else stitch_dir / "global_labels.npy"
if npy.exists():
    vol = np.load(npy, mmap_mode="r")        # memory-mapped: does not load the whole volume
    print(f"\n{npy.name}: shape={vol.shape}, dtype={vol.dtype}")

prov = stitch_dir / "label_provenance.csv"
if prov.exists():
    df = pd.read_csv(prov)
    suspects = df[df["seam_status"].isin(["seam", "corner"]) & (df["voxel_count"] > 0)]
    print(f"{len(df)} cells in the raw stitch, {len(suspects)} seam/corner cells to review")

## 11. Make the table usable with the Step 4 analysis

`cell_features.csv` is not identical to what Step 4 exports. Differences you should know about:

- It has **no `stack_name` column**. The analysis notebooks need one; the cell below adds it and writes `all_stacks_features.csv`.
- It is called `cell_features.csv`, not `features_<date>_<n>stacks.csv`.
- **Units** are micrometres only if you set `VOXEL_SIZE_UM`; otherwise voxels.
- **Orientation** (`FEAT_INVERT_Z`) is not mapped to Step 4's "Basal-down / Basal-up"; set that by hand where the analysis asks for it.
- The contact graph is `edges.csv`, with the same columns as Step 4's `<stack>_adjacency.csv`; the cell below also writes it under that name.
- **Not produced:** meshes, `step4_state.json`. The engine's labels **cannot be imported into the Step 4 Workbench**, so the Step 4 analysis does not run on this output unchanged.

In [ ]:
import shutil

STACK_NAME = INPUT_TIF.stem if INPUT_TIF is not None else OUTPUT_NAME

feat_csv = stitch_dir / "cell_features.csv"
if feat_csv.exists():
    table = pd.read_csv(feat_csv)
    table.insert(0, "stack_name", STACK_NAME)
    out_csv = stitch_dir / "all_stacks_features.csv"
    table.to_csv(out_csv, index=False)
    print(f"wrote {out_csv}  ({len(table)} cells)")
    print("Units :", "um" if VOXEL_SIZE_UM is not None else "VOXELS (VOXEL_SIZE_UM was None)")
    print("Orientation: set Basal-down / Basal-up yourself; FEAT_INVERT_Z =", FEAT_INVERT_Z)
else:
    print("No cell_features.csv yet - run section 9 first.")

edges = stitch_dir / "edges.csv"
if edges.exists():
    shutil.copyfile(edges, stitch_dir / f"{STACK_NAME}_adjacency.csv")
    print(f"wrote {STACK_NAME}_adjacency.csv")

## If you also have a nuclei segmentation (reconcile)

Next: run this notebook again for the nuclei (same input TIFF, a nuclei-trained model pair, `CHANNELS` on the nuclear stain, a different `OUTPUT_NAME`, and post-processing on so that the `*_clean` files exist). Then continue with `02_reconcile_cells_and_nuclei.ipynb` and `03_cleanup_to_consensus.ipynb`, which measure how well the cell and nuclei segmentations agree and make them consistent.